Business Question

Management asks:

Who are our top customers?

for the above question We need:

Revenue per customer
Number of orders
Average order value
Top 20 customers

in these analyatics we will be covering all the above tops which can answer the question 

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    sum,
    countDistinct,
    round,
    desc
)

In [6]:
spark = ( 
    SparkSession.builder
    .appName("Customer_Analytics")
    .master("local[*]")
    .getOrCreate()

)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/12 18:45:18 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.174 instead (on interface en0)
26/06/12 18:45:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 18:45:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/12 18:45:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
# reading Sales fact table 
sales_df=spark.read.parquet(
    "output/gold/sales_fact"
)
print("Total Sales Records:")
print(sales_df.count())
sales_df.printSchema()

Total Sales Records:
117601
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)



In [ ]:
# Revenue by Customer
#Calculates Customer Lifetime Value (CLV) metrics per customer:
# 1. Groups data by customer_id
# 2. Computes total revenue (sum of payments rounded to 2 decimals)
# 3. Counts unique orders placed (ignoring multiple items per order)
customer_revenue_df=(
    sales_df.groupBy("customer_id")
    .agg(
        round(sum("payment_value"),2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)
customer_revenue_df.show(5)

+--------------------+-------------+------------+
|         customer_id|total_revenue|total_orders|
+--------------------+-------------+------------+
|a836f6725983cd799...|       106.84|           1|
|72ecfc69f7d90359d...|       185.35|           1|
|dca00fb1b6171b713...|        69.64|           1|
|6f804e6a8f98ba0d1...|        47.59|           1|
|177fc7ae2e9f2151a...|        73.01|           1|
+--------------------+-------------+------------+
only showing top 5 rows


In [ ]:
# Average Order Value

customer_revenue_df=(
    customer_revenue_df.withColumn("avg_order_value")
)